#### This notebook uses for extracting data from uploaded files into final table loading into Unity Catalog

In [0]:
%pip install pdfplumber pytesseract pillow
# Restarting Python
dbutils.library.restartPython()

In [0]:

import pdfplumber
import pytesseract
from PIL import Image
import os
import pandas as pd
import shutil
from pathlib import Path
from datetime import datetime, timezone
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
dbutils.widgets.text("src_folder", "")
SRC_FOLDER = dbutils.widgets.get("src_folder")
#"/Volumes/atci_databricks_hackathon_2025/atci_clinical_transcript_ai_hackathon/raw_data/Bot_Uploaded_Files"
ARCHIVE_FOLDER = "/Volumes/atci_databricks_hackathon_2025/atci_clinical_transcript_ai_hackathon/raw_data/Bot_Archived_Files"

In [0]:
# EXTRACTING DATA
def extract_clinical_text_from_volume(src_folder: str):
    """
    Reads PDF files from a Unity Catalog Volume,
    extracts text (OCR fallback if needed),
    and returns structured records.
    """
    records = []
    # Resolve Databricks user once (avoid repeated Spark calls)
    current_user = spark.sql("SELECT current_user()").collect()[0][0]
    pdf_files = [
        f for f in os.listdir(src_folder)
        if f.lower().endswith(".pdf")
    ]

    if not pdf_files:
        print("⚠️ No PDF files found in the selected folder.")
        return records

    total_files = len(pdf_files)

    for idx, file_name in enumerate(pdf_files, start=1):
        filepath = os.path.join(src_folder, file_name)

        try:
            with pdfplumber.open(filepath) as pdf:
                for page_num, page in enumerate(pdf.pages, start=1):

                    # Extract text
                    text = page.extract_text()

                    # OCR fallback
                    if not text:
                        image = page.to_image(resolution=300).original
                        text = pytesseract.image_to_string(image)

                    records.append({
                        "file_name": file_name,
                        "page_number": page_num,
                        "page_text": text.strip() if text else None,
                        "update_datetime": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                        "updated_by": current_user
                    })
            # ✅ Move file ONLY after successful extraction
            archive_path = os.path.join(ARCHIVE_FOLDER, file_name)
            shutil.move(filepath, archive_path)

            print(f"✅ Completed: {filepath}")
            print(f"✅ Archived: {archive_path}")

        except Exception as e:
            print(f"❌ Failed to process {file_name}: {str(e)}")
    print(f"🎉 Extraction completed for {total_files} file(s).")
    return records


In [0]:
datestamp = datetime.now().strftime("%Y-%m-%d")
extracted_records = extract_clinical_text_from_volume(SRC_FOLDER)

In [0]:
# PARSING DATA USING LLM'S
PROMPT = """
You are a clinical data extractor.
Extract the following fields and return VALID JSON ONLY.
Do not include explanations or markdown.
Fields:
- patient_name
- mrn
- dob
- visit_date
- provider
- reason
- subjective
- objective
- bp_systolic
- bp_diastolic
- heart_rate
- assessment_and_plan

TEXT:
"""
schema = StructType([
    StructField("patient_name", StringType()),
    StructField("mrn", StringType()),
    StructField("dob", StringType()),
    StructField("visit_date", StringType()),
    StructField("provider", StringType()),
    StructField("reason", StringType()),
    StructField("subjective", StringType()),
    StructField("objective", StringType()),
    StructField("bp_systolic", IntegerType()),
    StructField("bp_diastolic", IntegerType()),
    StructField("heart_rate", IntegerType()),
    StructField("assessment_and_plan", StringType())
])


In [0]:
def extract_structured_fields_with_llm(records):
    """
    Takes extracted PDF page records (list of dicts),
    runs LLM extraction using AI_QUERY,
    and returns a structured Spark DataFrame.
    """
    if not records:
        return None
    # Convert records → Spark DF (Bronze)
    bronze_df = spark.createDataFrame(records)

    # -------------------------------
    # LLM Extraction
    # -------------------------------
    df_structured = bronze_df.withColumn(
        "llm_json",
        expr(f"""
            AI_QUERY(
                'databricks-meta-llama-3-3-70b-instruct',
                CONCAT('{PROMPT}', page_text)
            )
        """)
    )

    # -------------------------------
    # Clean LLM JSON (keep { ... })
    # -------------------------------
    df_structured = df_structured.withColumn(
        "llm_json_clean",
        trim(
            regexp_extract(
                col("llm_json"),
                r"(?s)(\{.*\})", 
                1
            )
        )
    )

    # -------------------------------
    # Parse JSON → Struct
    # -------------------------------
    df_structured = df_structured.withColumn(
        "extracted_json",
        from_json(
            col("llm_json_clean"),
            schema,
            {"mode": "PERMISSIVE"}
        )
    )

    # -------------------------------
    # Flatten → Final DataFrame
    # -------------------------------
    df_final = df_structured.select(
        "file_name",
        "page_number",
        "page_text",
        "llm_json",
        *[
            col(f"extracted_json.{field.name}").alias(field.name)
            for field in schema.fields
        ]
    )
    return df_final

In [0]:
%skip
df_structured = extract_structured_fields_with_llm(extracted_records)
df_structured.write.mode("append").format("delta").saveAsTable("atci_databricks_hackathon_2025.atci_clinical_transcript_ai_hackathon.clinial_data_extract_by_bot_silver")

In [0]:
%skip
%sql
DELETE FROM atci_databricks_hackathon_2025.atci_clinical_transcript_ai_hackathon.clinial_data_extract_by_bot_silver
WHERE EXISTS (
  SELECT 1
  FROM (
    SELECT mrn, visit_date, ROW_NUMBER() OVER (PARTITION BY mrn, visit_date ORDER BY visit_date) AS rn
    FROM atci_databricks_hackathon_2025.atci_clinical_transcript_ai_hackathon.clinial_data_extract_by_bot_silver
  ) t
  WHERE t.rn > 1
)

In [0]:

%skip
%sql
    DELETE 
    FROM atci_databricks_hackathon_2025.atci_clinical_transcript_ai_hackathon.clinial_data_extract_by_bot_silver

  --WHERE rn = 1

In [0]:
df_structured = extract_structured_fields_with_llm(extracted_records)
df_structured.createOrReplaceTempView("source_df")
spark.sql("""
MERGE INTO atci_databricks_hackathon_2025.atci_clinical_transcript_ai_hackathon.clinial_data_extract_by_bot_silver AS target
USING source_df AS source
ON target.mrn = source.mrn and target.visit_date = source.visit_date

WHEN MATCHED THEN
  UPDATE SET *

WHEN NOT MATCHED THEN
  INSERT *
""")